In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, ConfusionMatrixDisplay


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ И ПУТИ
# ======================
DATA = Path("../data/titanic.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
TEST_SIZE = 0.25

print("file exists:", DATA.exists())


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))


In [ ]:
data.info()
data.head()


In [ ]:
# Пропуски по столбцам — сколько NaN в каждом поле
print(data.isna().sum().sort_values(ascending=False))
print()
print("survived rate:", round(data["Survived"].mean(), 3))


In [ ]:
# Доля выживших по полу
class_counts = data.groupby("Sex")["Survived"].mean().sort_values()

# Установка размера фигуры
plt.figure(figsize=(8, 5))

# Столбчатая диаграмма с seaborn
sns.barplot(
    x=class_counts.index,   # Категории по оси X (пол)
    y=class_counts.values,  # Доля выживших по оси Y
    hue=class_counts.index,
    palette="viridis",
    legend=False,
)

# Настройка заголовка и подписей осей
plt.title("Выживаемость по полу", fontsize=14, pad=15)
plt.xlabel("Пол", fontsize=12)
plt.ylabel("Доля выживших", fontsize=12)

# Сетка по оси Y для читаемости
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Аннотации над столбцами
for i, val in enumerate(class_counts.values):
    plt.text(i, val + 0.02, f"{val:.2f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig(FIG / "titanic_survival_by_sex.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Подсчёт выживших / невыживших
surv_counts = data["Survived"].value_counts().sort_index()
labels_map = {0: "Не выжил", 1: "Выжил"}
colors = sns.color_palette("viridis", 2)

plt.figure(figsize=(7, 6))
ax = plt.subplot(111)

# Круговая диаграмма (donut)
wedges, texts = ax.pie(
    surv_counts,
    startangle=90,
    colors=colors,
    wedgeprops=dict(width=0.45, edgecolor="w", linewidth=1),
)

legend_labels = [
    f"{labels_map[i]} ({cnt} — {cnt / surv_counts.sum() * 100:.1f}%)"
    for i, cnt in surv_counts.items()
]
plt.legend(wedges, legend_labels, title="Survived", loc="center left", bbox_to_anchor=(1, 0.5))
plt.title("Распределение целевой переменной", fontsize=14, pad=15)

# Белый круг в центре
centre = plt.Circle((0, 0), 0.25, color="white")
ax.add_artist(centre)
plt.text(0, 0, f"Total:\n{surv_counts.sum()}", ha="center", va="center", fontsize=12)

plt.tight_layout()
plt.savefig(FIG / "titanic_survival_counts.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Выживаемость по классу каюты — ещё один быстрый взгляд
pclass_rate = data.groupby("Pclass")["Survived"].mean()

plt.figure(figsize=(8, 5))
sns.barplot(x=pclass_rate.index.astype(str), y=pclass_rate.values, palette="magma", hue=pclass_rate.index.astype(str), legend=False)
plt.title("Выживаемость по классу каюты (Pclass)", fontsize=14, pad=15)
plt.xlabel("Pclass", fontsize=12)
plt.ylabel("Доля выживших", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
# ======================
# 2. ПОДГОТОВКА ПРИЗНАКОВ
# ======================
# Числовые и категориальные признаки для baseline
num_cols = ["Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Pclass", "Sex", "Embarked"]
features = num_cols + cat_cols
target = "Survived"

df = data.dropna(subset=[target]).copy()
X = df[features]
y = df[target]

print("features:", features)
print("X shape:", X.shape)


In [ ]:
# ======================
# 3. TRAIN / TEST SPLIT
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y,  # сохранить долю Survived
)

print("train:", X_train.shape, "test:", X_test.shape)
print("train survived rate:", round(y_train.mean(), 3))
print("test  survived rate:", round(y_test.mean(), 3))


In [ ]:
# ======================
# 4. ПРЕПРОЦЕССИНГ
# ======================
# Числа: медиана + scaler; категории: moda + one-hot
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)


In [ ]:
# ======================
# 5. LOGISTIC REGRESSION
# ======================
logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

logreg.fit(X_train, y_train)
pred = logreg.predict(X_test)
proba = logreg.predict_proba(X_test)[:, 1]

print("LogReg accuracy", round(accuracy_score(y_test, pred), 3))
print("LogReg ROC-AUC", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, pred, digits=3))


In [ ]:
# ======================
# 6. RANDOM FOREST
# ======================
# Для дерева scaler не обязателен — оставляем только imputer / one-hot
rf_prep = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

rf = Pipeline([
    ("prep", rf_prep),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )),
])

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

print("RF accuracy", round(accuracy_score(y_test, pred_rf), 3))
print("RF ROC-AUC", round(roc_auc_score(y_test, proba_rf), 3))
print(classification_report(y_test, pred_rf, digits=3))


In [ ]:
# ======================
# 7. ВИЗУАЛИЗАЦИЯ: МАТРИЦА ОШИБОК
# ======================
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=axes[0], colorbar=False)
axes[0].set_title("LogReg — confusion matrix")

ConfusionMatrixDisplay.from_predictions(y_test, pred_rf, ax=axes[1], colorbar=False)
axes[1].set_title("RandomForest — confusion matrix")

plt.tight_layout()
plt.savefig(FIG / "titanic_confusion.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Сравнение ROC-AUC двух моделей на одном графике
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, proba, name="LogReg", ax=ax)
RocCurveDisplay.from_predictions(y_test, proba_rf, name="RandomForest", ax=ax)
ax.set_title("ROC-кривые: LogReg vs RandomForest", fontsize=13)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "titanic_roc_compare.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Краткий итог
print("Baseline LogReg AUC:", round(roc_auc_score(y_test, proba), 3))
print("RandomForest AUC:   ", round(roc_auc_score(y_test, proba_rf), 3))
print("Смотрим accuracy + ROC-AUC + report, не одну цифру.")
